In [0]:
# ---- Spark Performance Config ---- #
# r6id.12xlarge: 48 cores, 384GB RAM per node | 2-8 nodes = 96-384 cores

# Parallelism: 2-3x total cores for good task granularity
spark.conf.set("spark.default.parallelism", 768)
spark.conf.set("spark.sql.shuffle.partitions", 768)

# AQE: let Spark auto-optimize partitions, joins, and skew
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")

# Broadcast threshold: auto-broadcast tables under 100MB (coupon tables, maps)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")

# ANSI mode OFF: make cast() return null on bad values (matches Spark 3.4.1 / EMR behavior)
# Databricks Runtime 12+ defaults to ansi.enabled=true which makes cast() throw errors
spark.conf.set("spark.sql.ansi.enabled", "false")

# Legacy date parsing to match S3/EMR
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
%run ../../config/utils

In [0]:
"""Spark Job to carry out assignment of offers to members."""


from datetime import datetime
import sys
sys.path.append('..')
sys.path.append('../..')

from lib_assignment.assn_io import JobManager
from lib_assignment.assn_utils import env_path
from lib_assignment.campaign import Campaign
from lib_assignment.checks import check_execution_overwrite
from lib.iotools         import copy_file_to_s3, write_local_to_s3
from lib.utils             import apply_unionall


In [0]:
env = dbutils.widgets.get("environment")
allow_multi_long_tests = False if dbutils.widgets.get("allow_multi_long_tests")=="False" else True

In [0]:
# Optional: set checkpoint dir if sparkContext is accessible (single-user clusters).
# On shared clusters, sparkContext is blocked — localCheckpoint doesn't need it.
try:
    spark.sparkContext.setCheckpointDir("/dbfs/tmp/assign_offers")
except Exception:
    print("Shared cluster: sparkContext not available. localCheckpoint will be used (no checkpoint dir needed).")

In [0]:
name = "CampaignAssignment"
job = JobManager("assignment", "../config/config_template.yml", spark, output_vol, env)
truncate_history_base_dir = job.config.params.get(
    "truncate_history_base_dir",
    job.config.params["tmp_location"].rstrip("/") + "/assignment_truncate_history",
)
spark.conf.set(
    "assignment.truncate_history.base_dir",
    truncate_history_base_dir.rstrip("/") + "/assign_offers",
)
print(
    "truncate_history base dir: {}".format(
        spark.conf.get("assignment.truncate_history.base_dir")
    )
)

save_path = (
    "{output_dir}config_assign_{campaign}{run_name}{run_type}.yml".format(
        output_dir=job.config.paths["OUTPUT_DIR"],
        campaign=job.config.params["campaign"],
        run_name=job.config.params["run_name"],
        run_type=job.config.params["run_type"],
    )
)

if job.config.params["run_type"].lower() == "prod":
    paths_to_check = [
        save_path,
        job.config.paths["ASSIGNMENT_PATH"],
        job.config.paths["CONSTRUCTS_PATH"],
    ]

    check_execution_overwrite(job, paths_to_check=paths_to_check)

job.config.paths['RAW_MEMBER'] = silver_master_member_extended
job.config.paths['CUBE'] = fs_customer_cube_full
job.config.paths['ARTICLE_AH4_AH5_MAP'] = silver_master_item
job.config.paths['PRED_LIST'] = cf_prediction
job.config.paths['PROPENSITY'] = trip_spend_prediction

# # --- Save config ---- #
copy_file_to_s3(job.config.cfg_path, save_path, job.vol_base, job.env)
print("Config saved at {path}".format(path=env_path(save_path, output_vol, env)))



# ---- 1. Initialize campaign ---- #
print("1. Initializing...")
campaign = Campaign(job.config.params, job.config.paths, job.vol_base, job.env)



# ---- 2. ingest offer and member data ---- #
print("2. ingesting data sources...")
print("2.1 ingesting members")

member_data = campaign.ingest_member_data()
print("Member data ingested.")


print("2.2 ingesting offers")
assignment_pools, coupon_pools = campaign.ingest_offer_data()

print("2.3 calculate exposure")
offer_exposure, coupon_exposure = campaign.calculate_exposure()

# ---- 3. Frontfill ---- #
print("3. Calculate frontfill")
frontfill_offer = campaign.calculate_offers(
    member_data.select("MBRSHP_SID"),
    assignment_pools,
    coupon_pools,
    Campaign.FillType.FF,
    offer_exposure,
    coupon_exposure,
)


# ---- 4. Backfill ---- #
print("4. Calculate backfill")
backfill_offer = campaign.calculate_offers(
    member_data.select("MBRSHP_SID"),
    assignment_pools,
    coupon_pools,
    Campaign.FillType.BF,
    offer_exposure,
    coupon_exposure,
)


# ---- 5. Append fills ---- #
print("5. Append fills")
if frontfill_offer is not None and backfill_offer is not None:
    assignment = apply_unionall(frontfill_offer, backfill_offer)
elif frontfill_offer is not None:
    assignment = frontfill_offer
elif backfill_offer is not None:
    assignment = backfill_offer
else:
    raise Exception("Both frontfill and backfill are empty!")


# ---- 6. Fit coupons to the correct slot ---- #
print("6. Place coupons in the right position")
all_assignments = campaign.fit_coupons(assignment)

# ---- 7. Change layout for segment validation ---- #
print("7. Change layout for segment validation")
pivoted_assignment = campaign.pivot_assignment(all_assignments)
pivoted_assignment = member_data.join(pivoted_assignment, ["MBRSHP_SID"])

# ---- 8 Load past longitudinal mbrs ---- #
print("8. Add past longitudinal member eligibility")
pivoted_assignment = campaign.find_past_longitudinal_mbrs(
    pivoted_assignment
)

# ---- 9. Find eligible segments ---- #
print("9. Determining segment eligibility...")
eligible_assignments = campaign.find_eligible_segments(pivoted_assignment)

# ---- 10. Assign Members to cells ---- #
print("10. Assigning members to cells...")
member_data = campaign.assign_cells(eligible_assignments, allow_multi_long_tests)
member_data = campaign.unpivot_assignment(member_data)

# ---- 11. Reorder coupons based on user input ---- #
print("11. Map coupons based on layout mapping")
member_data = campaign.map_coupons(member_data)

# ---- 12. Reorder coupons based on user input ---- #
print("12. Reorder coupons based on user input")
member_data = campaign.sort_coupons(
    member_data, assignment_pools, coupon_pools
)

# ---- 13. Replace with null ---- #
print("13. Replace with null")
member_data = campaign.replace_slots_with_null(member_data)

# ---- 14. Build final assignment---- #
print("14. Generating assignment output...")
assigns, constructs = campaign.generate_output(member_data)

# ---- 15. Write Output ---- #
print("15. Writing output...")
job.data.add("assigns", assigns)
job.data.add("constructs", constructs)

if job.config.params["run_type"].lower() == "test":
    write_mode = "overwrite"
    job.data.write(
        "assigns",
        "ASSIGNMENT_PATH",
        mode=write_mode,
        singlefile=True,
        ftype="csv",
    )
    job.data.write(
        "constructs",
        "CONSTRUCTS_PATH",
        mode=write_mode,
        singlefile=True,
        ftype="parquet",
    )
else:
    write_mode = "overwrite"

    print("15.1 Writing true assignments.")
    job.data.write(
        "assigns",
        "ASSIGNMENT_PATH",
        mode=write_mode,
        ftype="csv",
        partitionby=32,
    )
    print("15.2 Writing all constructs.")
    job.data.write(
        "constructs",
        "CONSTRUCTS_PATH",
        mode=write_mode,
        ftype="parquet",
        partitionby=64,
    )
# # ---- 16. Write Log Output and Shut Down---- #
# print("16. Write Log Output and Shut Down...")

# log_line = {
#     "campaign": job.config.params["campaign"],
#     "date": datetime.now(),
#     "run_name": job.config.params["run_name"],
#     "run_type": job.config.params["run_type"],
#     "experiment": job.config.params["experiment"],
#     "assignment_date": job.config.params["assignment_date"],
#     "mail_list": job.config.paths["MAIL_LIST"],
#     "pred_list": job.config.paths["PRED_LIST"],
# }

# write_local_to_s3(log_line, job.config.paths["ASSIGN_LOG"])

print("done")
